# V11 decision suite — which LLM-embedding direction is worth a thesis on news recommendation?

Zero-shot **representation-level** benchmark on the exact **V10 protocol** (chronological validation split, `MINDsmall_dev` used once as test).
Every row is compared with the frozen embedding using a **paired 95% CI**, and split into **cold** (clicked article never seen in training) vs **warm**.

| Hypothesis | Technique (paper line) | Stage |
|---|---|---|
| H1 | history-as-text, instruction-aware query (LLM as user encoder; 2026 language-based user profiles) | `histquery` |
| H2 | co-click contrastive adaptation: residual heads, **LoRA / DoRA**, **Matryoshka (MRL)**, impression hard negatives (LLM2Rec, KDD'25) | `head`, `encoder` |
| H3 | entity / kNN **retrieval-augmented** article representations (RAG) | `graph` |
| H4 | **GraphRAG-lite**: Louvain communities + similar-user retrieval (K-RagRec, ACL'25 line) | `graph` |

**How to run (3 steps):** (1) Settings → Accelerator **GPU**, Internet **On**. (2) *Add Input* → a MIND dataset with **both** `MINDsmall_train` and `MINDsmall_dev` (e.g. `thinhhuynh3108/mindsmall`). (3) **Run All**.
Set `QUICK = True` first for a ~10 min smoke run on the real data.

The run ends with a **DECISION SUMMARY** (one representative per direction, chosen on validation, Bonferroni-adjusted), a **HEAD-TO-HEAD** table between the directions, and `test_arrays.npz` (per-impression metrics of every variant, so further paired analysis needs no re-run).

The first cell of results prints a **reproduction check**: the `frozen` row must match the V10 `bge_zeroshot` row (AUC 0.6241, nDCG@10 0.3909) — if it says MISMATCH, stop and fix before trusting anything else.

### 0. Dependencies

In [ ]:
%pip install -q peft sentence-transformers


### 1. Modules (written to the working directory)

In [ ]:
%%writefile data.py
"""Self-contained MIND-small data reader with chronological validation split."""
from __future__ import annotations

import math
import random
from collections import Counter
from itertools import groupby
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import torch

PAD = 0


@dataclass
class Batch:
    hist_title: torch.Tensor
    hist_cat: torch.Tensor
    hist_idx: torch.Tensor
    hist_mask: torch.Tensor
    cand_title: torch.Tensor
    cand_cat: torch.Tensor
    cand_idx: torch.Tensor
    cand_mask: torch.Tensor
    user_idx: torch.Tensor
    labels: torch.Tensor
    cand_pop: torch.Tensor

    def to(self, device):
        for name in self.__dataclass_fields__:
            setattr(self, name, getattr(self, name).to(device))
        return self


@dataclass
class MindData:
    news_title: np.ndarray
    news_cat: np.ndarray
    n_news: int
    n_users: int
    vocab_size: int
    n_cat: int
    title_len: int
    max_hist: int
    llm_emb: np.ndarray
    train_core: list = field(default_factory=list)
    validation: list = field(default_factory=list)
    test: list = field(default_factory=list)
    train_core_impressions: int = 0
    validation_impressions: int = 0
    test_impressions: int = 0
    train_days: list[str] = field(default_factory=list)
    validation_days: list[str] = field(default_factory=list)
    test_days: list[str] = field(default_factory=list)
    split_info: dict = field(default_factory=dict)
    _edges: list = field(default_factory=list)

    def _pad_hist(self, hist):
        hist = list(hist)[-self.max_hist:]
        mask = [True] * len(hist)
        while len(hist) < self.max_hist:
            hist.append(PAD)
            mask.append(False)
        return hist, mask

    def _gather(self, idxs):
        return self.news_title[idxs], self.news_cat[idxs]

    def collate_train(self, rows):
        H, L = self.max_hist, self.title_len
        C, B = 1 + len(rows[0][3]), len(rows)
        ht = np.zeros((B, H, L), np.int64)
        hc = np.zeros((B, H), np.int64)
        hi = np.zeros((B, H), np.int64)
        hm = np.zeros((B, H), bool)
        ct = np.zeros((B, C, L), np.int64)
        cc = np.zeros((B, C), np.int64)
        ci = np.zeros((B, C), np.int64)
        ui = np.zeros(B, np.int64)
        for b, (user, hist, pos, negs) in enumerate(rows):
            h, m = self._pad_hist(hist)
            hi[b], hm[b] = h, m
            ht[b], hc[b] = self._gather(h)
            cands = [pos] + list(negs)
            ci[b], ct[b], cc[b] = cands, *self._gather(cands)
            ui[b] = user
        return _to_batch(ht, hc, hi, hm, ct, cc, ci,
                         np.ones((B, C), bool), ui,
                         np.pad(np.ones((B, 1), np.float32), ((0, 0), (0, C - 1))),
                         np.zeros((B, C), np.float32))  # ponytail: no popularity for train rows, add when a trained model consumes cand_pop

    def collate_eval(self, rows):
        H, L = self.max_hist, self.title_len
        C, B = max(len(_eval_parts(r)[3]) for r in rows), len(rows)
        ht = np.zeros((B, H, L), np.int64)
        hc = np.zeros((B, H), np.int64)
        hi = np.zeros((B, H), np.int64)
        hm = np.zeros((B, H), bool)
        ct = np.zeros((B, C, L), np.int64)
        cc = np.zeros((B, C), np.int64)
        ci = np.zeros((B, C), np.int64)
        cm = np.zeros((B, C), bool)
        labels = np.zeros((B, C), np.float32)
        cp = np.zeros((B, C), np.float32)
        ui = np.zeros(B, np.int64)
        for b, row in enumerate(rows):
            _, user, hist, cands, labs, pop = _eval_parts(row)
            h, m = self._pad_hist(hist)
            hi[b], hm[b] = h, m
            ht[b], hc[b] = self._gather(h)
            n = len(cands)
            ci[b, :n], cm[b, :n], labels[b, :n], cp[b, :n] = cands, True, labs, pop
            if n:
                ct[b, :n], cc[b, :n] = self._gather(cands)
            ui[b] = user
        return _to_batch(ht, hc, hi, hm, ct, cc, ci, cm, ui, labels, cp)

    def build_adjacency(self):
        U, N = self.n_users, self.n_news
        rows, cols = [], []
        for u, n in self._edges:
            rows += [u, U + n]
            cols += [U + n, u]
        if not rows:
            rows, cols = [0], [0]
        idx = np.array([rows, cols], dtype=np.int64)
        vals = np.ones(idx.shape[1], np.float32)
        size = U + N
        deg = np.zeros(size, np.float64)
        np.add.at(deg, idx[0], vals)
        inv = np.zeros(size, np.float64)
        inv[deg > 0] = deg[deg > 0] ** -0.5
        norm = (inv[idx[0]] * inv[idx[1]]).astype(np.float32)
        return torch.sparse_coo_tensor(torch.from_numpy(idx), torch.from_numpy(norm), (size, size)).coalesce()

    @classmethod
    def from_mind(cls, train_dir, dev_dir, title_len=20, max_hist=50,
                  n_neg=4, min_word_freq=3, llm_dim=64,
                  llm_embeddings=None, dev_ratio=0.1, seed=0):
        import os
        from collections import Counter

        rng = np.random.default_rng(seed)

        def read_news(path):
            out = {}
            with open(path, encoding="utf-8") as f:
                for line in f:
                    p = line.rstrip("\n").split("\t")
                    if len(p) >= 5:
                        out[p[0]] = (p[1], p[3], p[4])
            return out

        news_rows = read_news(os.path.join(train_dir, "news.tsv"))
        if dev_dir and os.path.exists(os.path.join(dev_dir, "news.tsv")):
            news_rows.update(read_news(os.path.join(dev_dir, "news.tsv")))

        wf = Counter()
        for _, title, abstract in news_rows.values():
            wf.update((title + " " + abstract).lower().split())
        vocab = {"<pad>": PAD}
        for word, count in wf.items():
            if count >= min_word_freq:
                vocab[word] = len(vocab)
        cats = sorted({cat for cat, _, _ in news_rows.values()})
        cat_map = {cat: i + 1 for i, cat in enumerate(cats)}
        nid_map = {"<pad>": PAD}
        for nid in news_rows:
            nid_map[nid] = len(nid_map)
        n_news = len(nid_map)
        news_title = np.zeros((n_news, title_len), np.int64)
        news_cat = np.zeros(n_news, np.int64)
        for nid, (cat, title, abstract) in news_rows.items():
            words = (title + " " + abstract).lower().split()[:title_len]
            gi = nid_map[nid]
            news_title[gi, :len(words)] = [vocab.get(w, PAD) for w in words]
            news_cat[gi] = cat_map[cat]

        if llm_embeddings is None:
            llm_emb = np.zeros((n_news, llm_dim), np.float32)
        else:
            if not llm_embeddings:
                raise ValueError("empty BGE embedding map")
            dims = {np.asarray(v).reshape(-1).size for v in llm_embeddings.values()}
            if len(dims) != 1:
                raise ValueError("inconsistent embedding dimensions")
            dim = dims.pop()
            llm_emb = np.zeros((n_news, dim), np.float32)
            for nid, gi in nid_map.items():
                if nid != "<pad>" and nid in llm_embeddings:
                    llm_emb[gi] = np.asarray(llm_embeddings[nid], np.float32).reshape(-1)
            coverage = float((np.linalg.norm(llm_emb[1:], axis=1) > 0).mean())
            if coverage < 0.999 or not np.isfinite(llm_emb).all():
                raise ValueError(f"invalid BGE coverage={coverage:.4%}")
            print(f"  embedding coverage: {coverage:.3%} (PAD excluded)")
        llm_emb = (llm_emb / (np.linalg.norm(llm_emb, axis=1, keepdims=True) + 1e-8)).astype(np.float32)

        uid_map = {}
        def uidx(uid):
            if uid not in uid_map:
                uid_map[uid] = len(uid_map)
            return uid_map[uid]

        def time_of(value):
            value = value.strip()
            for fmt in ("%m/%d/%Y %H:%M:%S", "%m/%d/%Y %I:%M:%S %p", "%Y-%m-%d %H:%M:%S"):
                try:
                    return datetime.strptime(value, fmt)
                except ValueError:
                    pass
            raise ValueError(f"unparseable impression time: {value!r}")

        def read_behaviors(path):
            out = []
            with open(path, encoding="utf-8") as f:
                for line in f:
                    p = line.rstrip("\n").split("\t")
                    if len(p) < 5:
                        continue
                    impr_id, user_raw, timestamp, history_raw, impressions = p[:5]
                    hist = [nid_map[n] for n in history_raw.split() if n in nid_map] if history_raw else []
                    cands, labs = [], []
                    for item in impressions.split():
                        nid, lab = item.rsplit("-", 1)
                        if nid in nid_map:
                            cands.append(nid_map[nid])
                            labs.append(int(lab))
                    t = time_of(timestamp)
                    out.append((str(impr_id), uidx(user_raw), hist, cands, labs, t.date().isoformat(), t))
            return out

        train_raw = read_behaviors(os.path.join(train_dir, "behaviors.tsv"))
        if not dev_dir or os.path.abspath(dev_dir) == os.path.abspath(train_dir):
            raise ValueError("V10 requires separate MINDsmall_train and MINDsmall_dev")
        test_raw = read_behaviors(os.path.join(dev_dir, "behaviors.tsv"))
        if not train_raw or not test_raw:
            raise ValueError("MIND behaviors files are empty")
        pop = _online_log_ctr(train_raw + test_raw)
        train_raw = [r + (p,) for r, p in zip(train_raw, pop)]
        test_raw = [r + (p,) for r, p in zip(test_raw, pop[len(train_raw):])]

        days = sorted({row[5] for row in train_raw})
        target = max(1, int(np.ceil(len(train_raw) * dev_ratio)))
        val_days, count = [], 0
        for day in reversed(days):
            val_days.append(day)
            count += sum(row[5] == day for row in train_raw)
            if count >= target:
                break
        val_days = sorted(val_days)
        train_days = [d for d in days if d not in set(val_days)]
        if not train_days:
            raise ValueError("day split leaves no train_core day")
        train_raw_core = [r for r in train_raw if r[5] not in set(val_days)]
        val_raw = [r for r in train_raw if r[5] in set(val_days)]
        train_core_beh = [(u, h, c, labs) for _, u, h, c, labs, *_ in train_raw_core]
        validation = [(i, u, h, c, labs, p) for i, u, h, c, labs, _, _, p in val_raw]
        test = [(i, u, h, c, labs, p) for i, u, h, c, labs, _, _, p in test_raw]
        if max(r[5] for r in train_raw_core) >= min(r[5] for r in val_raw):
            raise AssertionError("chronological train/validation ordering violated")

        train_core, edges = [], []
        for user, hist, cands, labs in train_core_beh:
            edges.extend((user, n) for n in hist)
            pos = [c for c, lab in zip(cands, labs) if lab == 1]
            neg = [c for c, lab in zip(cands, labs) if lab == 0]
            if not pos or not neg:
                continue
            for p in pos:
                sampled = list(rng.choice(neg, size=n_neg, replace=len(neg) < n_neg))
                train_core.append((user, hist, p, sampled))
                edges.append((user, p))
        split_info = {
            "train_days": train_days,
            "validation_days": val_days,
            "test_days": sorted({r[5] for r in test_raw}),
            "train_day_min": min(train_days), "train_day_max": max(train_days),
            "validation_day_min": min(val_days), "validation_day_max": max(val_days),
            "target_validation_ratio": float(dev_ratio),
            "actual_validation_ratio": len(validation) / len(train_raw),
            "raw_train_impressions": len(train_raw),
            "train_core_impressions": len(train_core_beh),
            "validation_impressions": len(validation),
            "test_impressions": len(test),
        }
        print(f"  chronological split: {len(train_core_beh):,} train_core / "
              f"{len(validation):,} validation / {len(test):,} official test impressions")
        return cls(news_title, news_cat, n_news, len(uid_map), len(vocab), len(cat_map) + 1,
                   title_len, max_hist, llm_emb, train_core, validation, test,
                   len(train_core_beh), len(validation), len(test), train_days, val_days,
                   split_info["test_days"], split_info, edges)


def _online_log_ctr(rows):
    """Smoothed log-CTR of every candidate, counted only from impressions strictly earlier in time.

    Impressions sharing a timestamp are scored before any of them is counted, so no label leaks.
    """
    clicks, shown, out = Counter(), Counter(), [None] * len(rows)
    order = sorted(range(len(rows)), key=lambda k: rows[k][6])
    for _, group in groupby(order, key=lambda k: rows[k][6]):
        group = list(group)
        for k in group:
            out[k] = [math.log((clicks[n] + 1) / (shown[n] + 20)) for n in rows[k][3]]
        for k in group:
            for n, y in zip(rows[k][3], rows[k][4]):
                shown[n] += 1
                clicks[n] += y
    return out


def _eval_parts(row):
    if len(row) == 6:
        return row
    user, hist, cands, labs = row
    return "", user, hist, cands, labs, [0.0] * len(cands)


def _to_batch(*arrays):
    return Batch(*(torch.from_numpy(x) for x in arrays))


def iter_batches(rows, batch_size, collate, shuffle=False, seed=0):
    order = list(range(len(rows)))
    if shuffle:
        random.Random(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        yield collate([rows[i] for i in order[start:start + batch_size]])


In [ ]:
%%writefile metrics.py
"""Ranking metrics for news recommendation, computed per-impression.

Pure NumPy so there is no dependency beyond numpy. Definitions match the
Microsoft Recommenders / MIND leaderboard conventions (MRR, nDCG@k) plus a
tie-aware AUC.
"""
from __future__ import annotations

import numpy as np


def auc_score(labels: np.ndarray, scores: np.ndarray) -> float | None:
    """Area under ROC curve for one impression (tie-aware, rank based).

    Returns None if the impression has no positive or no negative (undefined),
    so the caller can skip it in the mean.
    """
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=np.float64)
    n_pos = int(labels.sum())
    n_neg = int(len(labels) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return None
    # Average ranks (1..n), ties share the mean rank -> Mann-Whitney U form.
    _, inv, counts = np.unique(scores, return_inverse=True, return_counts=True)
    cum = np.cumsum(counts)
    start = cum - counts
    avg = (start + cum + 1) / 2.0  # average 1-based rank per distinct value
    ranks = avg[inv]
    sum_ranks_pos = ranks[labels == 1].sum()
    return (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def mrr_score(labels: np.ndarray, scores: np.ndarray) -> float:
    order = np.argsort(scores)[::-1]
    y = np.asarray(labels)[order]
    rr = y / (np.arange(len(y)) + 1)
    denom = y.sum()
    return float(rr.sum() / denom) if denom > 0 else 0.0


def _dcg(labels: np.ndarray, scores: np.ndarray, k: int) -> float:
    order = np.argsort(scores)[::-1][:k]
    gains = (2 ** np.asarray(labels)[order] - 1).astype(np.float64)
    discounts = np.log2(np.arange(len(gains)) + 2)
    return float((gains / discounts).sum())


def ndcg_score(labels: np.ndarray, scores: np.ndarray, k: int) -> float:
    labels = np.asarray(labels)
    ideal = _dcg(labels, labels, k)
    if ideal == 0:
        return 0.0
    return _dcg(labels, scores, k) / ideal


def aggregate(impressions: list[tuple[np.ndarray, np.ndarray]]) -> dict[str, float]:
    """Mean metrics over a list of (labels, scores) impressions."""
    aucs, mrrs, n5, n10 = [], [], [], []
    for labels, scores in impressions:
        a = auc_score(labels, scores)
        if a is not None:
            aucs.append(a)
        mrrs.append(mrr_score(labels, scores))
        n5.append(ndcg_score(labels, scores, 5))
        n10.append(ndcg_score(labels, scores, 10))
    return {
        "auc": float(np.mean(aucs)) if aucs else 0.0,
        "mrr": float(np.mean(mrrs)) if mrrs else 0.0,
        "ndcg@5": float(np.mean(n5)) if n5 else 0.0,
        "ndcg@10": float(np.mean(n10)) if n10 else 0.0,
    }


In [ ]:
%%writefile rep_eval.py
"""Representation-level evaluation on the V10 protocol.

Zero-shot scorers (no learned user model) isolate the quality of the *item
embeddings*: user = mean of the clicked-history vectors, score = dot product with
the candidate vector.  Everything is vectorised in torch, so a full 73k-impression
pass takes seconds; the metric definitions are identical to ``metrics.py``
(tie-aware AUC, MRR over clicked items, nDCG@k) -- ``selftest.py`` asserts this.

Besides the usual means we keep:
* per-impression arrays -> paired confidence intervals between two variants;
* per-*positive* AUC contributions -> cold/warm slices (is the clicked article one
  the training data never saw?), the quantity LLM embeddings are meant to help.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F

import metrics as ref_metrics  # vendored V10 metrics.py (tie handling reference)

METRICS = ("auc", "mrr", "ndcg@5", "ndcg@10")


class EvalSet:
    """Padded, slate-size-sorted view of validation/test rows
    ``(impr_id, user, hist, cands, labels, pop)``."""

    def __init__(self, rows, max_hist=50, device="cpu", max_elems=3_000_000, max_batch=2048):
        self.rows, self.n = rows, len(rows)
        self.max_hist, self.device = max_hist, device
        self.max_elems, self.max_batch = max_elems, max_batch
        self.lens = np.array([len(r[3]) for r in rows], dtype=np.int64)
        self.order = np.argsort(self.lens, kind="stable")

    def __iter__(self):
        o, L, i = self.order, self.lens, 0
        while i < self.n:
            b = 1
            while (i + b < self.n and b < self.max_batch
                   and (b + 1) * max(int(L[o[i + b]]), 1) ** 2 <= self.max_elems):
                b += 1
            idx = o[i:i + b]
            i += b
            yield self._pack(idx)

    def _pack(self, idx):
        B, H = len(idx), self.max_hist
        C = max(int(self.lens[idx].max()), 1)
        hist = np.zeros((B, H), np.int64)
        hm = np.zeros((B, H), bool)
        cand = np.zeros((B, C), np.int64)
        cm = np.zeros((B, C), bool)
        lab = np.zeros((B, C), np.float32)
        user = np.zeros(B, np.int64)
        for k, r in enumerate(idx):
            _, u, h, c, y, _ = self.rows[r]
            h = list(h)[-H:]
            hist[k, :len(h)], hm[k, :len(h)] = h, True
            cand[k, :len(c)], cm[k, :len(c)], lab[k, :len(c)] = c, True, y
            user[k] = u
        t = lambda x: torch.from_numpy(x).to(self.device)
        return {"rows": idx, "hist": t(hist), "hist_mask": t(hm), "cand": t(cand),
                "cand_mask": t(cm), "labels": t(lab), "user": t(user)}


def batch_metrics(S, Y, M):
    """Per-impression metrics for padded scores ``S`` [B,C], labels ``Y``, mask ``M``."""
    B, C = S.shape
    S = torch.nan_to_num(S.float(), nan=0.0)
    Sm = S.masked_fill(~M, float("-inf"))
    pos = (Y > 0.5) & M
    neg = (Y <= 0.5) & M
    # exact tie-aware AUC through pairwise comparison (B*C*C is bounded by EvalSet)
    win = (Sm[:, :, None] > Sm[:, None, :]).float() + 0.5 * (Sm[:, :, None] == Sm[:, None, :]).float()
    n_neg = neg.sum(1).float()
    eq = (Sm[:, :, None] == Sm[:, None, :]) & M[:, :, None] & M[:, None, :]
    tie = (eq.sum((1, 2)) > M.sum(1)).cpu()  # more equalities than the diagonal alone
    pos_auc = (win * neg[:, None, :].float()).sum(2) / n_neg.clamp(min=1)[:, None]
    P = pos.sum(1).float()
    auc = (pos_auc * pos.float()).sum(1) / P.clamp(min=1)
    auc = torch.where((P > 0) & (n_neg > 0), auc, torch.full_like(auc, float("nan")))
    # ranking metrics
    order = Sm.argsort(dim=1, descending=True)
    ys = pos.float().gather(1, order)
    ranks = torch.arange(1, C + 1, device=S.device).float()[None, :]
    mrr = (ys / ranks).sum(1) / P.clamp(min=1)
    disc = 1.0 / torch.log2(ranks + 1.0)

    def ndcg(k):
        dcg = (ys[:, :k] * disc[:, :k]).sum(1)
        cum = torch.cumsum(disc[0, :min(k, C)], 0)
        kk = torch.clamp(P, max=min(k, C)).long()
        ideal = torch.where(kk > 0, cum[(kk - 1).clamp(min=0)], torch.zeros_like(dcg))
        return torch.where(ideal > 0, dcg / ideal.clamp(min=1e-12), torch.zeros_like(dcg))

    return {"auc": auc, "mrr": mrr, "ndcg@5": ndcg(5), "ndcg@10": ndcg(10),
            "pos_auc": pos_auc, "pos_mask": pos, "tie": tie, "scores": Sm}


@torch.no_grad()
def evaluate(score_fn, evalset, seen=None):
    """Run ``score_fn(batch) -> [B,C]`` over an EvalSet.

    ``seen`` (bool array over news ids): news the training data observed.  Used for
    the cold/warm slices, which are over *clicked* articles only.
    """
    arr = {k: np.full(evalset.n, np.nan, np.float64) for k in METRICS}
    pnews, pauc, prow = [], [], []
    for b in evalset:
        m = batch_metrics(score_fn(b), b["labels"], b["cand_mask"])
        r = b["rows"]
        for k in METRICS:
            arr[k][r] = m[k].cpu().numpy()
        if m["tie"].any():  # exact ties (e.g. users with empty history): defer to V10 metrics.py
            n_valid = b["cand_mask"].sum(1).cpu().numpy()
            sc = m["scores"].cpu().numpy()
            lb = b["labels"].cpu().numpy()
            for k in np.nonzero(m["tie"].numpy())[0]:
                n = int(n_valid[k])
                s_k, y_k = sc[k, :n].astype(np.float32), lb[k, :n]
                arr["mrr"][r[k]] = ref_metrics.mrr_score(y_k, s_k)
                arr["ndcg@5"][r[k]] = ref_metrics.ndcg_score(y_k, s_k, 5)
                arr["ndcg@10"][r[k]] = ref_metrics.ndcg_score(y_k, s_k, 10)
            evalset.n_tied = getattr(evalset, "n_tied", 0) + int(m["tie"].sum())
        pm = m["pos_mask"].cpu().numpy()
        pnews.append(b["cand"].cpu().numpy()[pm])
        pauc.append(m["pos_auc"].cpu().numpy()[pm])
        prow.append(np.repeat(r, pm.sum(1)))
    out = {"arr": arr,
           "pos_news": np.concatenate(pnews), "pos_auc": np.concatenate(pauc),
           "pos_row": np.concatenate(prow)}
    out["mean"] = {"auc": float(np.nanmean(arr["auc"])),
                   **{k: float(np.mean(arr[k])) for k in METRICS[1:]}}
    if seen is not None:
        cold = ~seen[out["pos_news"]]
        out["cold"] = cold
        out["cold_frac"] = float(cold.mean()) if len(cold) else float("nan")
        out["mean"]["cold_auc"] = float(out["pos_auc"][cold].mean()) if cold.any() else float("nan")
        out["mean"]["warm_auc"] = float(out["pos_auc"][~cold].mean()) if (~cold).any() else float("nan")
    return out


# ------------------------------------------------------------------ scorers
def user_vectors(E, b):
    m = b["hist_mask"].unsqueeze(-1).to(E.dtype)
    return (E[b["hist"]] * m).sum(1) / m.sum(1).clamp(min=1)


def mean_pool_scorer(E):
    """User = mean clicked-history vector; score = dot with the candidate (== cosine
    ranking for L2-normalised E).  Same scorer as the V10 ``bge_zeroshot`` baseline."""
    def f(b):
        return torch.einsum("bd,bcd->bc", user_vectors(E, b), E[b["cand"]])
    return f


# --------------------------------------------------------------- statistics
def ratio_ci(d, groups=None, z=1.96):
    """CI for the mean of ``d`` with (optional) clustered observations: positives of the
    same impression are correlated, so the SE uses per-impression sums (ratio estimator)."""
    d = np.asarray(d, np.float64)
    ok = ~np.isnan(d)
    d = d[ok]
    if len(d) < 2:
        return float("nan"), float("nan"), float("nan")
    if groups is None:
        m = d.mean()
        se = d.std(ddof=1) / math.sqrt(len(d))
        return float(m), float(m - z * se), float(m + z * se)
    g = np.asarray(groups)[ok]
    _, inv = np.unique(g, return_inverse=True)
    G = inv.max() + 1
    s = np.bincount(inv, weights=d, minlength=G)
    n = np.bincount(inv, minlength=G).astype(np.float64)
    m = s.sum() / n.sum()
    resid = s - m * n
    var = (G / max(G - 1, 1)) * (resid ** 2).sum() / (n.sum() ** 2)
    se = math.sqrt(var)
    return float(m), float(m - z * se), float(m + z * se)


def compare(res, ref, key="ndcg@10"):
    """Paired difference ``res - ref`` of an impression-level metric with a 95% CI."""
    return ratio_ci(res["arr"][key] - ref["arr"][key])


def compare_slice(res, ref, which="cold"):
    """Paired difference of the positive-level AUC on cold (or warm) clicked articles."""
    sel = res["cold"] if which == "cold" else ~res["cold"]
    if not sel.any():
        return float("nan"), float("nan"), float("nan")
    d = res["pos_auc"] - ref["pos_auc"]
    return ratio_ci(d[sel], groups=res["pos_row"][sel])


def verdict(lo, hi):
    if np.isnan(lo):
        return "n/a"
    return "BETTER" if lo > 0 else ("WORSE" if hi < 0 else "no sig. difference")


In [ ]:
%%writefile semgraph.py
"""Semantic-graph / retrieval-augmented representations ("GraphRAG-lite").

Why this exists: V10 showed the *click* graph (LightGCN) is noise on MIND-small -- a
fresh article has no click edge.  Here every variant is **content-only** (article
embeddings + the Wikidata entities MIND annotates on titles/abstracts, never click
labels), so it can still say something about *cold* articles:

* ``entity_view``   entity-grounded retrieval: an article is enriched with the context
                    vector of its entities (centroid of older articles mentioning them).
* ``knn_view``      dense retrieval: enrich with its k nearest older articles.
* ``communities``   GraphRAG-style global view: Louvain communities on the article kNN
                    graph; a user's history is summarised at community level.
* ``user_rag``      collaborative retrieval: add the clicks of similar *other* users.

Causality: the memory (what may be retrieved) is restricted to articles observed before
the evaluation period -- validation uses train_core only, test uses train_core+validation.
This is retrieval-augmented *representation* (no text generation); an LLM that reads the
retrieved context is the natural next step and is left out on purpose.
"""
from __future__ import annotations

import json
import os

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F


# --------------------------------------------------------------------- loading
def load_news_meta(train_dir, dev_dir):
    """Same article order as ``MindData.from_mind`` (global id = position + 1)."""
    by_id = {}
    for d in (train_dir, dev_dir):
        path = os.path.join(d, "news.tsv") if d else None
        if not path or not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                p = line.rstrip("\n").split("\t")
                if len(p) >= 5:
                    by_id[p[0]] = (p[1], p[3], p[4], p[6] if len(p) > 6 else "", p[7] if len(p) > 7 else "")
    ids = list(by_id)
    titles = [by_id[i][1] for i in ids]
    abstracts = [by_id[i][2] for i in ids]

    def wikidata(*cols):
        out = set()
        for c in cols:
            try:
                for e in json.loads(c) if c else []:
                    if e.get("WikidataId"):
                        out.add(e["WikidataId"])
            except (ValueError, TypeError, AttributeError):
                pass
        return sorted(out)

    ents = [wikidata(by_id[i][3], by_id[i][4]) for i in ids]
    return {"ids": ids, "titles": titles, "abstracts": abstracts,
            "text": [(t + ". " + a).strip() for t, a in zip(titles, abstracts)], "ents": ents}


def observed_masks(data):
    """Articles the *training* rows observed (history / clicked / sampled negative) and the
    articles seen in validation slates.  Index 0 (PAD) is always False."""
    n = data.n_news
    tr = np.zeros(n, bool)
    for _user, hist, pos, negs in data.train_core:
        tr[np.asarray(hist, dtype=np.int64)] = True
        tr[pos] = True
        tr[np.asarray(negs, dtype=np.int64)] = True
    va = np.zeros(n, bool)
    for _i, _u, hist, cands, _labs, _pop in data.validation:
        va[np.asarray(hist, dtype=np.int64)] = True
        va[np.asarray(cands, dtype=np.int64)] = True
    tr[0] = va[0] = False
    return tr, va


def entity_matrix(ents, n_news):
    """Binary article x entity incidence (row 0 = PAD stays empty)."""
    vocab, rows, cols = {}, [], []
    for i, es in enumerate(ents):
        for e in es:
            rows.append(i + 1)
            cols.append(vocab.setdefault(e, len(vocab)))
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n_news, max(len(vocab), 1)))
    return A


def _norm(x, eps=1e-8):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


# ---------------------------------------------------------------------- memory
class Memory:
    """Retrieval memory over a boolean mask of admissible (older) articles."""

    def __init__(self, E, A, mem, device="cpu"):
        self.E = np.ascontiguousarray(E, dtype=np.float32)
        self.A, self.mem, self.device = A, np.asarray(mem, bool), device
        self.mem_ids = np.nonzero(self.mem)[0]
        self.n = len(E)

    # --- entity-grounded retrieval
    def entity_view(self, beta):
        Am = sp.diags(self.mem.astype(np.float32)) @ self.A
        df = np.asarray(Am.sum(0)).ravel()
        ctx = _norm(np.asarray(Am.T @ self.E, np.float32))               # entity context vectors
        idf = np.log((len(self.mem_ids) + 1.0) / (df + 1.0)) * (df > 0)  # no memory -> no context
        R = np.asarray((self.A @ sp.diags(idf.astype(np.float32))) @ ctx, np.float32)
        has = np.linalg.norm(R, axis=1) > 0
        out = self.E.copy()
        out[has] = _norm(self.E[has] + beta * _norm(R[has]))
        return out, float(has[1:].mean())

    # --- dense retrieval
    def neighbours(self, rows, k, chunk=2048):
        """Top-k admissible neighbours (self excluded) for ``rows`` -> (idx [r,k], sim [r,k])."""
        dev = self.device
        Em = torch.from_numpy(self.E[self.mem_ids]).to(dev)
        pos_in_mem = np.full(self.n, -1, np.int64)
        pos_in_mem[self.mem_ids] = np.arange(len(self.mem_ids))
        rows = np.asarray(rows)
        nbr, sim = [], []
        for s in range(0, len(rows), chunk):
            r = rows[s:s + chunk]
            sc = torch.from_numpy(self.E[r]).to(dev) @ Em.T
            pm = pos_in_mem[r]
            hit = np.nonzero(pm >= 0)[0]
            if len(hit):
                sc[torch.from_numpy(hit).to(dev), torch.from_numpy(pm[hit]).to(dev)] = float("-inf")
            v, i = sc.topk(min(k, sc.shape[1]), dim=1)
            nbr.append(self.mem_ids[i.cpu().numpy()])
            sim.append(v.cpu().numpy())
        return np.concatenate(nbr), np.concatenate(sim)

    def knn_view(self, beta, k, rows=None):
        rows = np.arange(1, self.n) if rows is None else np.asarray(rows)
        nbr, sim = self.neighbours(rows, k)
        w = np.clip(sim, 0, None)
        w = w / (w.sum(1, keepdims=True) + 1e-8)
        agg = (self.E[nbr] * w[:, :, None]).sum(1)
        out = self.E.copy()
        out[rows] = _norm(self.E[rows] + beta * _norm(agg))
        return out

    # --- GraphRAG-style communities
    def communities(self, k=10, resolution=1.0, seed=0, max_nodes=80_000):
        """Louvain communities on the memory kNN graph; every other article joins its nearest
        community centroid.  Returns (labels [n], centroids [K,d])."""
        ids = self.mem_ids
        try:
            if len(ids) > max_nodes:
                raise RuntimeError("memory too large for networkx Louvain")
            import networkx as nx
            nbr, sim = self.neighbours(ids, k)
            G = nx.Graph()
            G.add_nodes_from(ids.tolist())
            for a, row, sw in zip(ids, nbr, sim):
                for b, w in zip(row, sw):
                    if w > 0 and a != b:
                        G.add_edge(int(a), int(b), weight=float(w))
            comms = nx.community.louvain_communities(G, weight="weight", resolution=resolution, seed=seed)
            lab_mem = np.zeros(len(ids), np.int64)
            where = {int(n): i for i, n in enumerate(ids)}
            for ci, c in enumerate(comms):
                for n in c:
                    lab_mem[where[n]] = ci
            method = "louvain"
        except Exception as exc:  # fall back to k-means centroids on the same embeddings
            from sklearn.cluster import MiniBatchKMeans
            K = int(max(8, min(500, len(ids) // 100)))
            lab_mem = MiniBatchKMeans(K, random_state=seed, n_init=3).fit_predict(self.E[ids])
            method = f"kmeans({exc.__class__.__name__})"
        K = int(lab_mem.max()) + 1
        cent = np.zeros((K, self.E.shape[1]), np.float32)
        np.add.at(cent, lab_mem, self.E[ids])
        cent = _norm(cent)
        labels = np.full(self.n, -1, np.int64)
        labels[ids] = lab_mem
        rest = np.nonzero(labels < 0)[0]
        rest = rest[rest > 0]
        if len(rest):
            labels[rest] = (self.E[rest] @ cent.T).argmax(1)
        return labels, cent, method


# --------------------------------------------------------------------- scorers
def community_scorer(E, labels, cent, lam, device="cpu"):
    """score = cos(user, cand) + lam * community-level affinity of the user's history."""
    Et = torch.from_numpy(np.ascontiguousarray(E, np.float32)).to(device)
    lab = torch.from_numpy(labels).to(device)
    C = torch.from_numpy(np.ascontiguousarray(cent, np.float32)).to(device)
    Cs, K = C @ C.T, len(cent)                                             # [K,K] community affinity

    def f(b):
        m = b["hist_mask"].unsqueeze(-1).float()
        u = (Et[b["hist"]] * m).sum(1) / m.sum(1).clamp(min=1)
        base = torch.einsum("bd,bcd->bc", u, Et[b["cand"]])
        hl = lab[b["hist"]].clamp(min=0)
        P = torch.zeros(hl.shape[0], K, device=Et.device).scatter_add_(1, hl, b["hist_mask"].float())
        P = P / P.sum(1, keepdim=True).clamp(min=1)                        # user's community histogram
        aff = P @ Cs                                                       # [B,K] affinity to each community
        return base + lam * aff.gather(1, lab[b["cand"]].clamp(min=0))
    return f


class UserMemory:
    """Profiles of users observed in training rows (history + clicks) and their click centroids."""

    def __init__(self, data, E, device="cpu"):
        U, N = data.n_users, data.n_news
        pu, pn, cu, cn = [], [], [], []
        for user, hist, pos, _negs in data.train_core:
            pu.extend([user] * len(hist)); pn.extend(hist)
            pu.append(user); pn.append(pos)
            cu.append(user); cn.append(pos)
        prof = sp.csr_matrix((np.ones(len(pu), np.float32), (pu, pn)), shape=(U, N)).sign()   # unique pairs
        clk = sp.csr_matrix((np.ones(len(cu), np.float32), (cu, cn)), shape=(U, N)).sign()
        self.profile = torch.from_numpy(_norm(np.asarray(prof @ E, np.float32))).to(device)
        self.clicks = torch.from_numpy(_norm(np.asarray(clk @ E, np.float32))).to(device)
        self.valid = (self.profile.norm(dim=1) > 0)
        self.device = device


def user_rag_scorer(E, um, gamma, device="cpu", k=20, chunk=256):
    """u' = u + gamma * mean(click centroids of the k most similar OTHER users)."""
    Et = torch.from_numpy(np.ascontiguousarray(E, np.float32)).to(device)
    prof, clicks, valid = um.profile.to(device), um.clicks.to(device), um.valid.to(device)

    def f(b):
        m = b["hist_mask"].unsqueeze(-1).float()
        u = (Et[b["hist"]] * m).sum(1) / m.sum(1).clamp(min=1)
        nb = torch.zeros_like(u)
        for s in range(0, u.shape[0], chunk):                              # bound the [chunk, n_users] similarity
            sim = F.normalize(u[s:s + chunk], dim=-1) @ prof.T
            sim = sim.masked_fill(~valid[None, :], float("-inf"))
            uid = b["user"][s:s + chunk].clamp(max=sim.shape[1] - 1).unsqueeze(1)
            sim.scatter_(1, uid, float("-inf"))                            # never retrieve the user itself
            top = sim.topk(min(k, sim.shape[1]), dim=1).indices
            nb[s:s + chunk] = clicks[top].mean(1)
        u2 = u + gamma * nb * (m.sum(1) > 0).float()                       # cold users keep their (empty) profile
        return torch.einsum("bd,bcd->bc", u2, Et[b["cand"]])
    return f


In [ ]:
%%writefile adapt.py
"""Contrastive adaptation of text embeddings to news recommendation.

Hypothesis (LLM2Rec, KDD'25 / LC-Rec / CoLLM line): a frozen text embedding knows what an
article is *about*, not which articles readers consume together.  We adapt it with a
co-click objective and ask whether the gain **transfers to articles never seen in
training** -- the case that matters for news.

Training signal (V10 ``train_core`` only, so nothing from validation/test leaks):
  anchor   = one of the reader's last ``k_anchor`` history articles
  positive = an article the reader then clicked
  hard neg = articles shown to the same reader in that impression but skipped
             (impression-aware negatives; MIND provides them for free)
  + in-batch negatives, with false-negative masking (same article id).

Adapters (all start as the identity => step 0 == the frozen control by construction):
  lin / mlp     residual head on precomputed vectors            (TAM-style, no encoder pass)
  lora / dora   low-rank adaptation of the encoder, in the loop (LoRA; DoRA = weight-decomposed)
Options: ``mrl`` Matryoshka loss over nested prefixes (MRL, NeurIPS'22; used in recsys by
fMRLRec/SMEC) and ``hardneg`` impression negatives.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------------- pairs
def build_pairs(data, k_anchor=3, max_pairs=200_000, seed=0):
    """(anchor, positive, hard-negatives) from ``data.train_core``."""
    rng = np.random.default_rng(seed)
    A, P, N = [], [], []
    seen = set()
    for _user, hist, pos, negs in data.train_core:
        if not len(hist):
            continue
        for a in hist[-k_anchor:]:
            key = (a, pos)
            if key in seen or a == pos:
                continue
            seen.add(key)
            A.append(a); P.append(pos); N.append(negs)
    A, P, N = np.array(A, np.int64), np.array(P, np.int64), np.array(N, np.int64)
    if len(A) > max_pairs:
        keep = rng.choice(len(A), max_pairs, replace=False)
        A, P, N = A[keep], P[keep], N[keep]
    return A, P, N


# -------------------------------------------------------------------- loss
def mrl_dims(d, scales=(64, 128, 256)):
    return [m for m in scales if m < d] + [d]


def contrastive_loss(a, p, p_idx, negs=None, tau=0.05, dims=None):
    """InfoNCE(anchor -> positive) with in-batch negatives (+ optional hard negatives),
    averaged over Matryoshka prefixes.  ``p_idx`` masks in-batch false negatives."""
    dims = dims or [a.shape[-1]]
    same = p_idx[:, None] == p_idx[None, :]
    same.fill_diagonal_(False)
    tgt = torch.arange(len(a), device=a.device)
    total = 0.0
    for m in dims:
        am, pm = F.normalize(a[:, :m], dim=-1), F.normalize(p[:, :m], dim=-1)
        logits = (am @ pm.T / tau).masked_fill(same, float("-inf"))
        if negs is not None:
            nm = F.normalize(negs[:, :, :m], dim=-1)
            logits = torch.cat([logits, torch.einsum("bd,bkd->bk", am, nm) / tau], dim=1)
        total = total + F.cross_entropy(logits, tgt)
    return total / len(dims)


# ------------------------------------------------------------------- heads
class ResLinear(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, d, bias=False)
        nn.init.zeros_(self.w.weight)

    def forward(self, x):
        return F.normalize(x + self.w(x), dim=-1)


class ResMLP(nn.Module):
    def __init__(self, d, h=None):
        super().__init__()
        self.a, self.b = nn.Linear(d, h or d), nn.Linear(h or d, d)
        nn.init.zeros_(self.b.weight)
        nn.init.zeros_(self.b.bias)

    def forward(self, x):
        return F.normalize(x + self.b(F.gelu(self.a(x))), dim=-1)


HEADS = {"lin": ResLinear, "mlp": ResMLP}


def train_head(E0, pairs, kind="lin", mrl=False, hardneg=False, epochs=3, bs=1024, lr=1e-3,
               tau=0.05, seed=0, device="cpu", val_fn=None, log=print):
    """Train a residual head on precomputed vectors.  ``val_fn(E) -> float`` (higher is
    better) selects the epoch; epoch 0 (= frozen control) competes, so the returned
    embedding is never worse than frozen on validation."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    E = torch.from_numpy(E0).to(device)
    head = HEADS[kind](E.shape[1]).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.0)
    A, P, N = pairs
    dims = mrl_dims(E.shape[1]) if mrl else None
    best = (val_fn(E0) if val_fn else -math.inf, E0.copy(), 0)
    for ep in range(1, epochs + 1):
        perm = rng.permutation(len(A))
        head.train()
        tot, nb = 0.0, 0
        for s in range(0, len(A), bs):
            ix = perm[s:s + bs]
            a, p = head(E[A[ix]]), head(E[P[ix]])
            n = head(E[N[ix]]) if hardneg else None
            loss = contrastive_loss(a, p, torch.from_numpy(P[ix]).to(device), n, tau, dims)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        head.eval()
        with torch.no_grad():
            Ea = head(E).cpu().numpy()
        score = val_fn(Ea) if val_fn else float(ep)          # no validation signal: keep the last epoch
        log(f"      [{kind}{'+mrl' if mrl else ''}{'+hn' if hardneg else ''}] epoch {ep} loss={tot / max(nb, 1):.4f} val={score:.4f}")
        if score > best[0]:
            best = (score, Ea, ep)
    return best[1], {"best_epoch": best[2], "val": float(best[0])}


# ----------------------------------------------------------- encoder in loop
POOLING_BY_NAME = (("qwen3-embedding", "last"), ("bge", "cls"), ("e5", "mean"), ("gte", "mean"), ("minilm", "mean"))


def guess_pooling(name):
    low = name.lower()
    for key, mode in POOLING_BY_NAME:
        if key in low:
            return mode
    return "mean"


class HFEncoder:
    """Plain Hugging Face encoder with explicit pooling (so LoRA/DoRA training and
    inference share one code path and no ST-version drift)."""

    def __init__(self, name, pooling=None, max_len=128, device="cpu", trust_remote_code=False, prefix=""):
        from transformers import AutoModel, AutoTokenizer
        self.name, self.max_len, self.device, self.prefix = name, max_len, device, prefix
        self.pooling = pooling or guess_pooling(name)
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=trust_remote_code)
        if self.pooling == "last":
            self.tok.padding_side = "left"
        self.model = AutoModel.from_pretrained(name, trust_remote_code=trust_remote_code).to(device)
        self.amp = device == "cuda"

    def _pool(self, out, attn):
        h = out.last_hidden_state
        if self.pooling == "cls":
            return h[:, 0]
        if self.pooling == "last":
            return h[:, -1]
        m = attn.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

    def embed(self, texts):
        batch = self.tok([self.prefix + t for t in texts], padding=True, truncation=True,
                         max_length=self.max_len, return_tensors="pt").to(self.device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp):
            out = self.model(**{k: v for k, v in batch.items() if k in ("input_ids", "attention_mask", "token_type_ids")})
        return self._pool(out, batch["attention_mask"]).float()

    @torch.no_grad()
    def encode(self, texts, bs=128):
        was = self.model.training
        self.model.eval()
        order = np.argsort([len(t) for t in texts])
        vecs = np.zeros((len(texts), self.model.config.hidden_size), np.float32)
        for s in range(0, len(texts), bs):
            ix = order[s:s + bs]
            vecs[ix] = F.normalize(self.embed([texts[i] for i in ix]), dim=-1).cpu().numpy()
        self.model.train(was)
        return vecs

    def attach_peft(self, mode="lora", r=16, dropout=0.05):
        from peft import LoraConfig, get_peft_model
        names = {n.split(".")[-1] for n, _ in self.model.named_modules()}
        targets = ["q_proj", "v_proj"] if "q_proj" in names else ["query", "value"]
        cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=dropout, target_modules=targets,
                         use_dora=(mode == "dora"), bias="none")
        self.model = get_peft_model(self.model, cfg)
        return sum(p.numel() for p in self.model.parameters() if p.requires_grad)


def check_equivalence(enc, st_encode, texts, tol=0.99):
    """Guard against pooling/prefix mistakes: HFEncoder must agree with the reference
    (Sentence-Transformers) embeddings of the same model."""
    a, b = enc.encode(texts), st_encode(texts)
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)
    return float(cos.min()), float(cos.mean()), bool(cos.min() >= tol)


def train_encoder(enc, texts, pairs, E0, mode="lora", mrl=False, hardneg=True, epochs=1, bs=32,
                  lr=2e-4, tau=0.05, seed=0, r=16, val_ids=None, val_fn=None, max_steps=None, log=print):
    """Contrastive LoRA/DoRA training with the encoder in the loop.

    After every epoch only the articles needed for validation are re-encoded (``val_ids``);
    epoch 0 (= the frozen embedding) competes, and the adapter weights of the best epoch are
    restored before one final pass over *all* articles.  Returns ``(E_final, info)``."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n_train = enc.attach_peft(mode, r)
    log(f"      [{mode} r={r}] trainable params: {n_train:,}")
    named = [(n, p) for n, p in enc.model.named_parameters() if p.requires_grad]
    params = [p for _, p in named]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=enc.amp)
    A, P, N = pairs
    dims = mrl_dims(E0.shape[1]) if mrl else None
    best_score, best_ep, best_state = (val_fn(E0) if val_fn else -math.inf), 0, None
    step, t0 = 0, time.time()
    for ep in range(1, epochs + 1):
        enc.model.train()
        perm = rng.permutation(len(A))
        tot, nb = 0.0, 0
        for s in range(0, len(A), bs):
            ix = perm[s:s + bs]
            ids = np.concatenate([A[ix], P[ix]] + ([N[ix].ravel()] if hardneg else []))
            uniq, inv = np.unique(ids, return_inverse=True)
            z = enc.embed([texts[i] for i in uniq])
            inv = torch.from_numpy(inv).to(z.device)
            b = len(ix)
            a, p = z[inv[:b]], z[inv[b:2 * b]]
            n = z[inv[2 * b:]].view(b, -1, z.shape[-1]) if hardneg else None
            loss = contrastive_loss(a, p, torch.from_numpy(P[ix]).to(z.device), n, tau, dims)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            tot += loss.item(); nb += 1; step += 1
            if step % 100 == 0:
                log(f"        step {step} loss={tot / nb:.4f} ({time.time() - t0:.0f}s)")
            if max_steps and step >= max_steps:
                break
        if val_fn is not None and val_ids is not None:
            Ea = E0.copy()
            Ea[val_ids] = enc.encode([texts[i] for i in val_ids])
            score = val_fn(Ea)
        else:
            score = float(ep)                                              # no validation signal: keep the last epoch
        log(f"      [{mode}] epoch {ep} loss={tot / max(nb, 1):.4f} val={score:.4f}")
        if score > best_score:
            best_score, best_ep = score, ep
            best_state = {n: p.detach().clone() for n, p in named}
        if max_steps and step >= max_steps:
            break
    info = {"best_epoch": best_ep, "val": float(best_score), "steps": step, "trainable": n_train}
    if best_ep == 0:
        return E0.copy(), info                                             # adaptation did not beat frozen on validation
    with torch.no_grad():
        for n, p in named:
            p.copy_(best_state[n])
    E = E0.copy()
    E[1:] = enc.encode(texts[1:])
    return E, info


In [ ]:
%%writefile suite.py
"""V11 decision suite: which LLM-embedding direction is worth a thesis on news recommendation?

Every variant is scored zero-shot (user = mean of clicked-history vectors) so the table
measures *representation quality*, on the exact V10 protocol (chronological validation,
MINDsmall_dev as the one-shot test).  Hyper-parameters are picked on validation, the test
set is touched once per selected configuration, and every comparison to the control comes
with a paired 95% CI plus a cold-article slice (clicked articles the training rows never saw).

Hypotheses
  H1 history-as-text, instruction-aware query beats the mean of item vectors      (LLM-as-user-encoder)
  H2 co-click contrastive adaptation of the embedding (head / LoRA / DoRA,
     +Matryoshka, +impression hard negatives) beats the frozen embedding          (LLM2Rec-style)
  H3 entity / kNN retrieval-augmented article representations help fresh articles  (RAG)
  H4 GraphRAG-style community context and similar-user retrieval help             (GraphRAG-lite)
"""
from __future__ import annotations

import argparse
import json
import os
import time
import traceback

import numpy as np
import torch

import adapt
import rep_eval as R
import semgraph as G
from data import MindData

EQUIV_REFERENCE = None  # callable(texts)->np.ndarray of reference embeddings (e.g. Sentence-Transformers), set by the notebook

TASK = "Given the headlines a reader recently clicked, retrieve the news article they would most likely read next"


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ helpers
def query_prefix(name, fmt="auto", instruct=True):
    low = name.lower()
    if fmt == "auto":
        fmt = "qwen3" if "qwen3" in low else "bge" if "bge" in low else "e5" if "e5" in low else "plain"
    if fmt == "qwen3":
        return f"Instruct: {TASK}\nQuery:" if instruct else ""
    if fmt == "bge":
        return "Represent this sentence for searching relevant passages: " if instruct else ""
    if fmt == "e5":
        return "query: "
    return ""


def avg_results(rs):
    """Average several seeds of the same variant (per-impression arrays and slice values)."""
    if len(rs) == 1:
        return rs[0]
    out = dict(rs[0])
    out["arr"] = {k: np.nanmean([r["arr"][k] for r in rs], axis=0) for k in R.METRICS}
    out["pos_auc"] = np.mean([r["pos_auc"] for r in rs], axis=0)
    m = {"auc": float(np.nanmean(out["arr"]["auc"])), **{k: float(np.mean(out["arr"][k])) for k in R.METRICS[1:]}}
    if "cold" in out:
        c = out["cold"]
        m["cold_auc"] = float(out["pos_auc"][c].mean()) if c.any() else float("nan")
        m["warm_auc"] = float(out["pos_auc"][~c].mean()) if (~c).any() else float("nan")
    out["mean"] = m
    return out


class Suite:
    def __init__(self, data, meta, E0, device, work, seen_tr, seen_va):
        self.data, self.meta, self.E0, self.device, self.work = data, meta, E0, device, work
        self.seen_tr, self.seen_va = seen_tr, seen_va
        self.es = {"val": R.EvalSet(data.validation, device=device), "test": R.EvalSet(data.test, device=device)}
        self.records, self.cache = [], {}

    def tensor(self, E):
        return torch.from_numpy(np.ascontiguousarray(E, dtype=np.float32)).to(self.device)

    def run(self, scorer, split):
        return R.evaluate(scorer, self.es[split], self.seen_tr)

    def val_ndcg(self, E):
        return self.run(R.mean_pool_scorer(self.tensor(E)), "val")["mean"]["ndcg@10"]

    def record(self, name, family, cfg, val, test, ref_name, extra=None):
        """Store a row; Δ vs the reference is computed on the test split with paired CIs."""
        ref = self.cache[ref_name]["test"] if ref_name in self.cache else None
        row = {"name": name, "family": family, "cfg": cfg, "ref": ref_name,
               "val_ndcg@10": val["mean"]["ndcg@10"], "val_auc": val["mean"]["auc"], **test["mean"],
               "cold_frac": test.get("cold_frac"), **(extra or {})}
        if ref is not None and name != ref_name:
            row["d_ndcg@10"] = R.compare(test, ref, "ndcg@10")
            row["d_auc"] = R.compare(test, ref, "auc")
            row["d_cold_auc"] = R.compare_slice(test, ref, "cold")
            row["d_warm_auc"] = R.compare_slice(test, ref, "warm")
            row["verdict"] = R.verdict(row["d_ndcg@10"][1], row["d_ndcg@10"][2])
        self.records.append(row)
        self.cache[name] = {"val": val, "test": test}
        d = row.get("d_ndcg@10")
        log(f"  {name:34s} val nDCG@10={row['val_ndcg@10']:.4f} | test AUC={row['auc']:.4f} nDCG@10={row['ndcg@10']:.4f}"
            + (f" | Δ={d[0]:+.4f} [{d[1]:+.4f},{d[2]:+.4f}] {row['verdict']}" if d else "")
            + (f" | cold AUC={row.get('cold_auc', float('nan')):.4f}" if "cold_auc" in row else ""))
        return row

    # ---- generic grid: choose the config on validation, report it on test once
    def select(self, name, family, configs, make, ref_name, extra=None):
        best = None
        for cfg in configs:
            v = self.run(make(cfg, "val"), "val")
            log(f"    {name} {cfg}: val nDCG@10={v['mean']['ndcg@10']:.4f}")
            if best is None or v["mean"]["ndcg@10"] > best[0]["mean"]["ndcg@10"]:
                best = (v, cfg)
        v, cfg = best
        t = self.run(make(cfg, "test"), "test")
        return self.record(name, family, cfg, v, t, ref_name, extra)

    def export(self, name, E, model_name):
        path = os.path.join(self.work, f"emb_{name}.npz")
        np.savez(path, ids=np.array(self.meta["ids"]), vecs=np.asarray(E[1:], np.float32), model_name=np.array(model_name))
        return path


# ------------------------------------------------------------------- stages
def stage_controls(S, args):
    log("\n== controls ==")
    E = S.tensor(S.E0)
    v, t = S.run(R.mean_pool_scorer(E), "val"), S.run(R.mean_pool_scorer(E), "test")
    S.record("frozen", "control", {}, v, t, "frozen")
    if args.repro:
        auc_ref, nd_ref = (float(x) for x in args.repro.split(","))
        ok = abs(t["mean"]["auc"] - auc_ref) < 0.003 and abs(t["mean"]["ndcg@10"] - nd_ref) < 0.003
        log(f"  reproduction check vs V10 bge_zeroshot (AUC {auc_ref}, nDCG@10 {nd_ref}): "
            f"{'MATCH' if ok else 'MISMATCH - check input text / truncation / embedding file'}")
    if t.get("cold_frac") is not None:
        log(f"  clicked articles that are cold (unseen in training rows): {t['cold_frac']:.1%}")


def stage_graph(S, args):
    log("\n== H3/H4 retrieval-augmented representations (content-only, causal memory) ==")
    A = G.entity_matrix(S.meta["ents"], S.data.n_news)
    n_with = int((np.asarray(A.sum(1)).ravel() > 0).sum())
    log(f"  entity coverage: {n_with:,}/{S.data.n_news - 1:,} articles carry Wikidata entities")
    mem = {"val": S.seen_tr, "test": S.seen_tr | S.seen_va}
    memo = {sp: G.Memory(S.E0, A, mem[sp], S.device) for sp in ("val", "test")}
    if n_with:
        S.select("entity_rag", "H3-RAG", [{"beta": b} for b in args.betas],
                 lambda c, sp: R.mean_pool_scorer(S.tensor(memo[sp].entity_view(c["beta"])[0])), "frozen")
    nn = {sp: memo[sp].neighbours(np.arange(1, S.data.n_news), max(args.knn_k)) for sp in ("val", "test")}

    def knn_view(c, sp):
        nbr, sim = nn[sp]
        k = c["k"]
        w = np.clip(sim[:, :k], 0, None)
        w = w / (w.sum(1, keepdims=True) + 1e-8)
        E = S.E0.copy()
        for s0 in range(0, len(nbr), 8192):                       # chunked: [rows,k,d] would be ~GBs at k=20
            sl = slice(s0, s0 + 8192)
            agg = (S.E0[nbr[sl, :k]] * w[sl, :, None]).sum(1)
            E[1 + s0:1 + s0 + len(agg)] = G._norm(S.E0[1 + s0:1 + s0 + len(agg)] + c["beta"] * G._norm(agg))
        return E
    S.select("knn_rag", "H3-RAG", [{"k": k, "beta": b} for k in args.knn_k for b in args.betas],
             lambda c, sp: R.mean_pool_scorer(S.tensor(knn_view(c, sp))), "frozen")

    comm = {}
    for sp in ("val", "test"):
        t0 = time.time()
        comm[sp] = memo[sp].communities(k=10, resolution=args.louvain_res, seed=0)
        log(f"  communities[{sp}]: {comm[sp][1].shape[0]} via {comm[sp][2]} ({time.time() - t0:.0f}s)")
    S.select("community_graphrag", "H4-GraphRAG", [{"lam": l} for l in args.lams],
             lambda c, sp: G.community_scorer(S.E0, comm[sp][0], comm[sp][1], c["lam"], S.device), "frozen")
    um = G.UserMemory(S.data, S.E0, S.device)
    S.select("user_rag", "H4-GraphRAG", [{"gamma": g} for g in args.gammas],
             lambda c, sp: G.user_rag_scorer(S.E0, um, c["gamma"], S.device), "frozen")


def _seed_runs(S, name, family, cfg, train_fn, ref, seeds, trunc=None):
    """Train a stochastic adapter over seeds -> one averaged row; Matryoshka truncation reported."""
    vals, tests, Es, infos = [], [], [], []
    for sd in range(seeds):
        E, info = train_fn(sd)
        Et = S.tensor(E)
        vals.append(S.run(R.mean_pool_scorer(Et), "val"))
        tests.append(S.run(R.mean_pool_scorer(Et), "test"))
        Es.append(E); infos.append(info)
    extra = {"seeds": seeds, "best_epochs": [i["best_epoch"] for i in infos],
             "ndcg10_per_seed": [t["mean"]["ndcg@10"] for t in tests]}
    if seeds > 1:
        extra["ndcg10_std"] = float(np.std(extra["ndcg10_per_seed"]))
    for m in (trunc or []):
        Em = Es[0][:, :m].copy()
        Em = Em / (np.linalg.norm(Em, axis=1, keepdims=True) + 1e-8)
        extra[f"ndcg10@{m}d"] = S.run(R.mean_pool_scorer(S.tensor(Em)), "test")["mean"]["ndcg@10"]
    S.record(name, family, cfg, avg_results(vals), avg_results(tests), ref, extra)
    return Es[0]


def stage_head(S, args):
    log("\n== H2 contrastive co-click adaptation: residual heads on frozen vectors ==")
    pairs = adapt.build_pairs(S.data, max_pairs=args.pairs)
    log(f"  {len(pairs[0]):,} (anchor, clicked, hard-negative) pairs from train_core")
    d = S.E0.shape[1]
    trunc = [m for m in (64, 128) if m < d]
    ref_t = S.cache["frozen"]["test"]
    for m in trunc:  # how good is plain truncation of the frozen vector? (the thing MRL should fix)
        Em = S.E0[:, :m] / (np.linalg.norm(S.E0[:, :m], axis=1, keepdims=True) + 1e-8)
        log(f"  frozen truncated to {m}d: test nDCG@10={S.run(R.mean_pool_scorer(S.tensor(Em)), 'test')['mean']['ndcg@10']:.4f}"
            f" (full {ref_t['mean']['ndcg@10']:.4f})")
    best = None
    for kind in ("lin", "mlp"):
        for tag, mrl, hn in (("", False, False), ("+hn", False, True), ("+hn+mrl", True, True)):
            name = f"head_{kind}{tag}"
            fn = lambda sd, kind=kind, mrl=mrl, hn=hn: adapt.train_head(
                S.E0, pairs, kind, mrl, hn, epochs=args.epochs_head, seed=sd, device=S.device,
                val_fn=S.val_ndcg, log=lambda *a: None)
            E = _seed_runs(S, name, "H2-adapt", {"kind": kind, "mrl": mrl, "hardneg": hn}, fn, "frozen", args.seeds, trunc)
            r = S.records[-1]
            if best is None or r["val_ndcg@10"] > best[0]:
                best = (r["val_ndcg@10"], name, E)
    log(f"  best head on validation: {best[1]}  -> exported {S.export(best[1], best[2], args.encoder)}")


def stage_encoder(S, args):
    log("\n== H2 contrastive co-click adaptation: LoRA / DoRA with the encoder in the loop ==")
    texts = [""] + S.meta["text"]
    val_ids = np.unique(np.concatenate([np.asarray(r[2], np.int64).ravel() for r in S.data.validation]
                                        + [np.asarray(r[3], np.int64) for r in S.data.validation]))
    pairs = adapt.build_pairs(S.data, max_pairs=args.pairs_enc)
    log(f"  {len(pairs[0]):,} pairs, {len(val_ids):,} validation articles re-encoded per epoch")
    enc0 = adapt.HFEncoder(args.encoder, max_len=args.max_len, device=S.device, prefix=args.doc_prefix)
    Ef = np.zeros_like(S.E0)
    Ef[1:] = enc0.encode(S.meta["text"], bs=args.enc_bs)
    Et = S.tensor(Ef)
    S.record("frozen_hf", "control", {"max_len": args.max_len}, S.run(R.mean_pool_scorer(Et), "val"),
             S.run(R.mean_pool_scorer(Et), "test"), "frozen_hf")
    del enc0
    for mode, mrl in (("lora", True), ("dora", True), ("lora", False)):
        name = f"{mode}{'+mrl' if mrl else ''}+hn"
        enc = adapt.HFEncoder(args.encoder, max_len=args.max_len, device=S.device, prefix=args.doc_prefix)
        t0 = time.time()
        E, info = adapt.train_encoder(enc, texts, pairs, Ef, mode=mode, mrl=mrl, hardneg=True,
                                      epochs=args.epochs_enc, bs=args.bs_enc, r=args.lora_r, val_ids=val_ids,
                                      val_fn=S.val_ndcg, max_steps=args.enc_steps, log=log)
        Et = S.tensor(E)
        S.record(name, "H2-adapt", {"mode": mode, "mrl": mrl, "r": args.lora_r}, S.run(R.mean_pool_scorer(Et), "val"),
                 S.run(R.mean_pool_scorer(Et), "test"), "frozen_hf",
                 {"trainable": info["trainable"], "steps": info["steps"], "train_s": time.time() - t0})
        S.export(name, E, args.encoder)
        del enc
        if S.device == "cuda":
            torch.cuda.empty_cache()


def stage_histquery(S, args):
    log("\n== H1 history-as-text, instruction-aware query ==")
    titles = [""] + S.meta["titles"]
    K = args.hist_k
    keys, rowq = {}, {"val": [], "test": []}
    for sp, rows in (("val", S.data.validation), ("test", S.data.test)):
        for r in rows:
            h = tuple(r[2][-K:])
            rowq[sp].append(keys.setdefault(h, len(keys)) if h else -1)
    uniq = sorted(keys, key=keys.get)
    log(f"  {len(uniq):,} unique histories (last {K} headlines) to encode")
    Et = S.tensor(S.E0)
    for instruct in (True, False):
        pre = query_prefix(args.encoder, args.query_format, instruct)
        enc = adapt.HFEncoder(args.encoder, max_len=args.q_max_len, device=S.device, prefix=pre)
        Q = np.zeros((len(uniq) + 1, S.E0.shape[1]), np.float32)
        t0 = time.time()
        Q[:-1] = enc.encode(["; ".join(titles[i] for i in h) for h in uniq], bs=args.enc_bs)
        log(f"  encoded queries ({'instruct' if instruct else 'plain'}) in {time.time() - t0:.0f}s, prefix={pre[:40]!r}")
        Qt = S.tensor(Q)
        rq = {sp: torch.tensor(np.where(np.array(rowq[sp]) < 0, len(uniq), rowq[sp]), device=S.device) for sp in rowq}
        mp = R.mean_pool_scorer(Et)

        def mk(c, sp, rq=rq, Qt=Qt, mp=mp):
            def f(b):
                s = torch.einsum("bd,bcd->bc", Qt[rq[sp][torch.as_tensor(b["rows"], device=S.device)]], Et[b["cand"]])
                return s + c["w"] * mp(b) if c["w"] else s
            return f
        tag = "instr" if instruct else "plain"
        S.select(f"histq_{tag}", "H1-text-user", [{"w": 0.0}], mk, "frozen")
        S.select(f"histq_{tag}+meanpool", "H1-text-user", [{"w": w} for w in (0.5, 1.0, 2.0)], mk, "frozen")
        del enc


# --------------------------------------------------------------------- output
def fmt_ci(t, digits=4):
    return f"{t[0]:+.{digits}f} [{t[1]:+.{digits}f}, {t[2]:+.{digits}f}]" if t else "-"


def write_table(S, work):
    cols = ["variant", "family", "val nDCG@10", "test AUC", "MRR", "nDCG@5", "nDCG@10", "Δ nDCG@10 [95% CI]",
            "cold AUC", "Δ cold AUC [95% CI]", "warm AUC", "verdict"]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in S.records:
        lines.append("| " + " | ".join([
            r["name"], r["family"], f"{r['val_ndcg@10']:.4f}", f"{r['auc']:.4f}", f"{r['mrr']:.4f}",
            f"{r['ndcg@5']:.4f}", f"{r['ndcg@10']:.4f}", fmt_ci(r.get("d_ndcg@10")),
            f"{r.get('cold_auc', float('nan')):.4f}", fmt_ci(r.get("d_cold_auc")),
            f"{r.get('warm_auc', float('nan')):.4f}", r.get("verdict", "control")]) + " |")
    md = "\n".join(lines)
    with open(os.path.join(work, "results.md"), "w", encoding="utf-8") as f:
        f.write("# V11 decision suite (zero-shot representation quality, MINDsmall_dev test)\n\n" + md + "\n")
    with open(os.path.join(work, "results.json"), "w", encoding="utf-8") as f:
        json.dump(S.records, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    return md


def summarize(S):
    """One representative per direction, chosen on VALIDATION (never on test), then compared with the
    control.  The verdict is also Bonferroni-adjusted for the number of directions being compared,
    because picking 'the best of 4 directions' is a multiple-comparison decision."""
    from statistics import NormalDist
    log("\n================ DECISION SUMMARY ================")
    by_h = {}
    for r in S.records:
        if r["family"] != "control" and "verdict" in r:
            by_h.setdefault(r["family"], []).append(r)
    z_adj = NormalDist().inv_cdf(1 - 0.05 / (2 * max(len(by_h), 1)))
    for fam, rs in by_h.items():
        best = max(rs, key=lambda r: r["val_ndcg@10"])             # selection on validation only
        d, c = best["d_ndcg@10"], best.get("d_cold_auc")
        se = (d[2] - d[1]) / (2 * 1.96)
        adj = R.verdict(d[0] - z_adj * se, d[0] + z_adj * se)
        log(f"{fam:16s} rep={best['name']:26s} (best of {len(rs)} by val) Δ nDCG@10 {fmt_ci(d)} -> {best['verdict']}"
            f" | Bonferroni(m={len(by_h)}): {adj}"
            + (f" | Δ cold AUC {fmt_ci(c)}" if c and not np.isnan(c[0]) else ""))
    log("Rule: pursue a direction only if its Bonferroni-adjusted verdict is BETTER; prefer the ones that also win on the "
        "cold slice (where LLM embeddings should matter) and whose gain is practically meaningful (>~0.005 nDCG@10).\n"
        "Unadjusted 'BETTER' on one of ~20 rows can be chance: with all effects null, about 1 in 20 rows would still pass.")
    return [max(rs, key=lambda r: r["val_ndcg@10"]) for rs in by_h.values()]


def head_to_head(S, reps):
    """Direction-vs-direction comparison on the test split (paired over impressions), Bonferroni over the pairs.
    The control comparison above says whether a direction helps at all; this says which direction is better."""
    from itertools import combinations
    from statistics import NormalDist
    pairs = list(combinations(reps, 2))
    out = []
    if not pairs:
        return out
    z_adj = NormalDist().inv_cdf(1 - 0.05 / (2 * len(pairs)))
    log(f"\n-- HEAD-TO-HEAD between directions (test, paired). CI shown is unadjusted 95%; the verdict is Bonferroni-adjusted over {len(pairs)} pairs --")
    for a, b in pairs:
        ra, rb = S.cache[a["name"]]["test"], S.cache[b["name"]]["test"]
        d, c = R.compare(ra, rb, "ndcg@10"), R.compare_slice(ra, rb, "cold")
        se = (d[2] - d[1]) / (2 * 1.96)
        lo, hi = d[0] - z_adj * se, d[0] + z_adj * se
        word = f"{a['family']} > {b['family']}" if lo > 0 else (f"{b['family']} > {a['family']}" if hi < 0 else "not distinguishable")
        out.append({"a": a["name"], "b": b["name"], "d_ndcg@10": d, "d_cold_auc": c, "verdict": word})
        log(f"  {a['family']}:{a['name']} vs {b['family']}:{b['name']}  Δ nDCG@10 {fmt_ci(d)} | Δ cold AUC {fmt_ci(c)} -> {word}")
    return out


def save_arrays(S):
    """Per-impression metrics of every variant (test split) so any further paired analysis needs no re-run."""
    out = {"impr_ids": np.array([r[0] for r in S.data.test])}
    for name, c in S.cache.items():
        t = c["test"]
        for k in R.METRICS:
            out[f"{name}__{k}"] = t["arr"][k].astype(np.float32)
        out[f"{name}__pos_auc"] = t["pos_auc"].astype(np.float32)
    first = next(iter(S.cache.values()))["test"]
    out["pos_news"], out["pos_row"], out["cold"] = first["pos_news"], first["pos_row"], first["cold"]
    path = os.path.join(S.work, "test_arrays.npz")
    np.savez_compressed(path, **out)
    return path


# ------------------------------------------------------------------------ main
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--mind-train", required=True)
    ap.add_argument("--mind-dev", required=True)
    ap.add_argument("--work", default="v11_out")
    ap.add_argument("--encoder", default="BAAI/bge-small-en-v1.5")
    ap.add_argument("--news-emb", default=None, help="npz from V10 llm_embed.py (exact V10 control)")
    ap.add_argument("--doc-prefix", default="")
    ap.add_argument("--query-format", default="auto")
    ap.add_argument("--max-len", type=int, default=256)
    ap.add_argument("--q-max-len", type=int, default=384)
    ap.add_argument("--stages", default="controls,graph,head,encoder,histquery")
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--repro", default=None, help='published V10 control "auc,ndcg10" (bge_zeroshot: 0.6241,0.3909)')
    ap.add_argument("--pairs", type=int, default=200_000)
    ap.add_argument("--pairs-enc", type=int, default=100_000)
    ap.add_argument("--epochs-head", type=int, default=3)
    ap.add_argument("--epochs-enc", type=int, default=1)
    ap.add_argument("--bs-enc", type=int, default=32)
    ap.add_argument("--enc-steps", type=int, default=1500)
    ap.add_argument("--enc-bs", type=int, default=128)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--hist-k", type=int, default=15)
    ap.add_argument("--louvain-res", type=float, default=1.0)
    ap.add_argument("--betas", type=float, nargs="+", default=[0.25, 0.5, 1.0])
    ap.add_argument("--knn-k", type=int, nargs="+", default=[5, 20])
    ap.add_argument("--lams", type=float, nargs="+", default=[0.25, 0.5, 1.0, 2.0])
    ap.add_argument("--gammas", type=float, nargs="+", default=[0.25, 0.5, 1.0])
    args = ap.parse_args(argv)

    os.makedirs(args.work, exist_ok=True)
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else ("cpu" if args.device == "auto" else args.device)
    log(f"device={device} encoder={args.encoder} stages={args.stages}")
    meta = G.load_news_meta(args.mind_train, args.mind_dev)
    if args.news_emb:
        z = np.load(args.news_emb, allow_pickle=True)
        emb = {str(i): v for i, v in zip(z["ids"], z["vecs"])}
        log(f"E0 <- {args.news_emb} ({len(emb):,} vectors)")
    else:
        cache = os.path.join(args.work, "news_emb_E0.npz")
        if os.path.exists(cache) and str(np.load(cache)["model_name"]) == f"{args.encoder}|{args.max_len}|{args.doc_prefix}":
            z = np.load(cache, allow_pickle=True)
            vecs = z["vecs"]
        else:
            enc = adapt.HFEncoder(args.encoder, max_len=args.max_len, device=device, prefix=args.doc_prefix)
            t0 = time.time()
            vecs = enc.encode(meta["text"], bs=args.enc_bs)
            if EQUIV_REFERENCE is not None:  # catches a wrong pooling / EOS / prefix convention up front
                ref = np.asarray(EQUIV_REFERENCE(meta["text"][:64]), np.float32)
                cos = (vecs[:64] * ref).sum(1) / (np.linalg.norm(vecs[:64], axis=1) * np.linalg.norm(ref, axis=1) + 1e-8)
                log(f"E0 guard: HFEncoder vs reference embeddings, min cos={cos.min():.4f} mean={cos.mean():.4f}")
                if cos.min() < 0.99:
                    raise RuntimeError("HFEncoder disagrees with the reference encoder (pooling/EOS/prefix) - fix before running")
            np.savez(cache, ids=np.array(meta["ids"]), vecs=vecs, model_name=np.array(f"{args.encoder}|{args.max_len}|{args.doc_prefix}"))
            log(f"E0 encoded with {args.encoder} in {time.time() - t0:.0f}s")
            del enc
        emb = dict(zip(meta["ids"], vecs))
    data = MindData.from_mind(args.mind_train, args.mind_dev, llm_embeddings=emb)
    assert data.n_news - 1 == len(meta["ids"]), "article order mismatch between loader and metadata"
    seen_tr, seen_va = G.observed_masks(data)
    S = Suite(data, meta, data.llm_emb, device, args.work, seen_tr, seen_va)
    stages = {"controls": stage_controls, "graph": stage_graph, "head": stage_head,
              "encoder": stage_encoder, "histquery": stage_histquery}
    wanted = [s for s in args.stages.split(",") if s]
    if "controls" not in wanted:
        wanted.insert(0, "controls")
    for s in wanted:
        t_stage = time.time()
        try:
            stages[s](S, args)
            log(f"-- stage '{s}' finished in {time.time() - t_stage:.0f}s")
        except Exception:  # one failing stage must not discard the others
            log(f"\n!! stage '{s}' failed:\n{traceback.format_exc()}")
        write_table(S, args.work)
    md = write_table(S, args.work)
    log("\n" + md)
    S.reps = summarize(S)
    S.h2h = head_to_head(S, S.reps)
    with open(os.path.join(args.work, "head_to_head.json"), "w", encoding="utf-8") as f:
        json.dump(S.h2h, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    log(f"per-impression arrays -> {save_arrays(S)}")
    return S


if __name__ == "__main__":
    main()


### 2. Configuration

In [ ]:
import os
QUICK = bool(int(os.environ.get("V11_QUICK", "0")))        # True = ~10 min smoke run on the real data
ENCODER = os.environ.get("V11_ENCODER", "BAAI/bge-small-en-v1.5")
RUN_QWEN3 = bool(int(os.environ.get("V11_QWEN3", "0")))    # also run the cheap stages with Qwen3-Embedding-0.6B
QWEN3 = "Qwen/Qwen3-Embedding-0.6B"
STAGES = "controls,graph,head,encoder,histquery"
MIND_TRAIN = os.environ.get("V11_MIND_TRAIN")               # leave None -> auto-detect under /kaggle/input
MIND_DEV = os.environ.get("V11_MIND_DEV")
DEVICE = os.environ.get("V11_DEVICE", "auto")
WORK = "/kaggle/working/v11_out" if os.path.isdir("/kaggle/working") else "v11_out"
MAX_LEN = 256


### 3. Locate MIND (needs a train + dev pair from the SAME dataset)

In [ ]:
import glob
if not MIND_TRAIN:
    hits = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/news.tsv", recursive=True)})
    trs = [d for d in hits if "train" in d.lower()]
    dvs = [d for d in hits if "dev" in d.lower() or "valid" in d.lower()]
    for t in trs:
        sib = [d for d in dvs if os.path.dirname(d) == os.path.dirname(t)]
        if sib:
            MIND_TRAIN, MIND_DEV = t, sib[0]
            break
assert MIND_TRAIN and MIND_DEV and MIND_TRAIN != MIND_DEV, (
    "Attach a MIND dataset that has BOTH MINDsmall_train and MINDsmall_dev (e.g. thinhhuynh3108/mindsmall).")
count = lambda p: sum(1 for _ in open(p, encoding="utf-8"))
print("train:", MIND_TRAIN, "| behaviors:", f"{count(MIND_TRAIN + '/behaviors.tsv'):,}")
print("dev  :", MIND_DEV, "| behaviors:", f"{count(MIND_DEV + '/behaviors.tsv'):,}  (MIND-small: ~157k / ~73k)")


### 4. Run the suite (encoder: BGE-small). Each stage is isolated: a failing stage is reported and the rest still run.

In [ ]:
import sys
sys.path.insert(0, os.getcwd())
import torch, suite

def reference_encoder(name, max_len):
    """Sentence-Transformers reference used to guard against wrong pooling/EOS/prefix conventions."""
    try:
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer(name, device="cuda" if torch.cuda.is_available() else "cpu")
        st.max_seq_length = max_len
        return lambda texts: st.encode(list(texts), normalize_embeddings=True)
    except Exception as e:
        print("reference encoder unavailable, skipping the pooling guard:", repr(e)[:120])
        return None

QUICK_ARGS = ["--seeds", "1", "--pairs", "20000", "--pairs-enc", "3000", "--enc-steps", "60", "--epochs-head", "1",
              "--betas", "0.5", "--knn-k", "10", "--lams", "1.0", "--gammas", "0.5", "--bs-enc", "16"]
repro = ["--repro", "0.6241,0.3909"] if "bge-small-en-v1.5" in ENCODER else []
suite.EQUIV_REFERENCE = reference_encoder(ENCODER, MAX_LEN)
S = suite.main(["--mind-train", MIND_TRAIN, "--mind-dev", MIND_DEV, "--work", WORK, "--encoder", ENCODER,
                "--stages", STAGES, "--device", DEVICE, "--max-len", str(MAX_LEN), *repro, *(QUICK_ARGS if QUICK else [])])


### 5. (Optional) same cheap stages with Qwen3-Embedding-0.6B — a stronger LLM-based encoder (set `RUN_QWEN3 = True`)

In [ ]:
if RUN_QWEN3:
    suite.EQUIV_REFERENCE = reference_encoder(QWEN3, MAX_LEN)
    S3 = suite.main(["--mind-train", MIND_TRAIN, "--mind-dev", MIND_DEV, "--work", WORK + "_qwen3", "--encoder", QWEN3,
                     "--stages", "controls,graph,head,histquery", "--device", DEVICE, "--max-len", str(MAX_LEN),
                     "--q-max-len", "512", "--enc-bs", "32", *(QUICK_ARGS if QUICK else [])])


### 6. Results

In [ ]:
print(open(os.path.join(WORK, "results.md"), encoding="utf-8").read())
print("exported embeddings (usable with V10 bench.py --news-emb):", [f for f in os.listdir(WORK) if f.startswith("emb_")])
